## Building Deep Research agent through agent orchestration


In [99]:
import asyncio
import os
import smtplib
from email.message import EmailMessage

from agents.model_settings import ModelSettings
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from rich.console import Console

# from rich.markdown import Markdown
from agents import Agent, Runner, WebSearchTool, function_tool, trace


In [118]:
console = Console()
OPENAI_MODEL = "gpt-5.4-nano"
HOW_MANY_SEARCHES = 5

load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

### Agent 1 : Search Agent


In [101]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [102]:
search_agent = Agent(
    name="Search Agent",
    tools=tools,
    instructions=INSTRUCTIONS,
    model_settings=settings,
    model=OPENAI_MODEL,
)

### Agent 2: Planner Agent


In [103]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Provide reasoning why this search is important to the query")
    query: str = Field(description="The search term to use for the web search")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="List of web search results")

In [104]:
INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

In [105]:
planner_agent = Agent(
    name="Planner Agent", instructions=INSTRUCTIONS, model=OPENAI_MODEL, output_type=WebSearchPlan
)

### Agent 3 - Writer Agent


In [106]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""

In [107]:
class ReportData(BaseModel):
    short_summary: str = Field(description="A Short cummary of the research content")
    report: str = Field(description="The fresearch report in HTML format")
    follow_up_questions: str = Field(description="Follow-up questions on the research tiem")

In [108]:
writer_agent = Agent(
    name="Writer Agent", model=OPENAI_MODEL, instructions=INSTRUCTIONS, output_type=ReportData
)

### Agent 4 - Email Agent


In [109]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SERVER = os.getenv("EMAIL_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

if not EMAIL_ADDRESS or not EMAIL_SERVER or not EMAIL_APP_PASSWORD:
    print("Email configuration not received")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SERVER and EMAIL_APP_PASSWORD

print("Email setup completed") if USE_EMAIL else print("Email setup failed")


# Test the email
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)
        print("Email sent")


Email setup completed


In [110]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_email(subject, text_body, html_body)

In [111]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

In [112]:
email_agent = Agent(
    name="Email Agent", instructions=INSTRUCTIONS, model=OPENAI_MODEL, tools=[send_email_tool]
)

### Orchestration with the four agents


In [113]:
# coroutine to search the web
async def search(item: WebSearchItem) -> str:
    input_message = f"Search for {item.query}"
    result = await Runner.run(search_agent, input_message)

    return result.final_output

In [114]:
# Coroutine to plan for the search
async def search_plan(query: str) -> str:
    console.print("Starting Planning...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    console.print(f"Starting {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)

    return results

In [115]:
# Coroutine to write a report
async def write_report(query: str) -> ReportData:
    search_results = await search_plan(query)
    console.print("Starting reporting...")
    result = await Runner.run(
        writer_agent, f"Original query: {query}\n Summarized search results: {search_results}"
    )

    return result.final_output

In [ ]:
# Coroutine to send email
async def send_email_with_report(query: str) -> str:
    report_result = await write_report(query)
    console.print("Starting to send email...")
    result = await Runner.run(email_agent, report_result.model_dump_json())

    return result.final_output

In [117]:
query = "Best AI agents in 2026"

with trace("Research trace"):
    await send_email_with_report(query)

Starting Planning...

Starting 20 searches

Starting reporting...

Srarting to send email...

Email sent
